In [ ]:
!pip install langchain langchain-community langchain-openai faiss-cpu openai \
  fastapi uvicorn pypdf beautifulsoup4 requests tiktoken pyngrok unstructured langchain_huggingface

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 10.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.8/453.8 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.9 MB/s eta 0:00:00
   ━━━━━━

In [ ]:
# @title 2. Mount Drive
from google.colab import drive
import os

drive.mount('/content/drive')

# Create working directories
!mkdir -p /content/k8s_docs
!mkdir -p /content/drive/MyDrive/k8s_rag_data

print("✅ Setup complete!")



MessageError: Error: credential propagation was unsuccessful

In [ ]:
# @title 3. Download REAL Kubernetes Documentation (Using known URLs)
import requests
import os
import time

def fetch_real_k8s_docs(output_dir="/content/k8s_docs"):
    """Download actual documentation pages with real content"""
    os.makedirs(output_dir, exist_ok=True)

    # These are actual documentation pages with real content
    doc_urls = [
        # Deployment and Scaling
        "https://kubernetes.io/docs/concepts/workloads/controllers/deployment/",
        "https://kubernetes.io/docs/tasks/run-application/scale-application/",
        "https://kubernetes.io/docs/tasks/run-application/horizontal-pod-autoscale/",

        # ConfigMaps and Configuration
        "https://kubernetes.io/docs/concepts/configuration/configmap/",
        "https://kubernetes.io/docs/tasks/configure-pod-container/configure-pod-configmap/",

        # Pods
        "https://kubernetes.io/docs/concepts/workloads/pods/",
        "https://kubernetes.io/docs/concepts/workloads/pods/pod-lifecycle/",

        # Services and Networking
        "https://kubernetes.io/docs/concepts/services-networking/service/",
        "https://kubernetes.io/docs/tasks/access-application-cluster/service-access-application-cluster/",

        # Storage
        "https://kubernetes.io/docs/concepts/storage/persistent-volumes/",
        "https://kubernetes.io/docs/tasks/configure-pod-container/configure-persistent-volume-storage/",

        # Kubectl
        "https://kubernetes.io/docs/reference/kubectl/overview/",
        "https://kubernetes.io/docs/reference/kubectl/cheatsheet/",

        # Namespaces
        "https://kubernetes.io/docs/concepts/overview/working-with-objects/namespaces/",

        # Jobs
        "https://kubernetes.io/docs/concepts/workloads/controllers/job/",
    ]

    downloaded = 0
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

    for url in doc_urls:
        try:
            print(f"Downloading ({downloaded+1}/{len(doc_urls)}): {url}")
            response = requests.get(url, timeout=15, headers=headers)

            # Parse HTML to extract main content
            from bs4 import BeautifulSoup
            soup = BeautifulSoup(response.text, 'html.parser')

            # Find main content area (common Kubernetes docs structure)
            main_content = soup.find('main') or soup.find('article')

            if main_content:
                # Remove navigation elements
                for nav in main_content.find_all(['nav', 'aside', '.td-sidebar', '.td-toc', '.feedback']):
                    nav.decompose()

                # Get text content
                text = main_content.get_text(separator='\n', strip=True)

                # Only save if we got substantial content
                if len(text) > 500:
                    # Create filename from URL path
                    path = url.replace('https://kubernetes.io/docs/', '').replace('/', '_')
                    filename = f"{path}.txt"
                    filepath = os.path.join(output_dir, filename)

                    with open(filepath, 'w', encoding='utf-8') as f:
                        f.write(f"Source: {url}\n")
                        f.write("="*60 + "\n\n")
                        f.write(text)

                    downloaded += 1
                    print(f"  ✅ Saved {len(text)} chars")
                else:
                    print(f"  ⚠️  Content too short ({len(text)} chars)")
            else:
                print(f"  ⚠️  Could not find main content")

            time.sleep(0.5)  # Be nice to the server

        except Exception as e:
            print(f"  ❌ Error: {e}")

    print(f"\n✅ Downloaded {downloaded} documentation pages to {output_dir}")
    return downloaded

# Run the download
doc_count = fetch_real_k8s_docs()
print(f"\n📁 Files saved:")
for f in os.listdir('/content/k8s_docs')[:10]:
    filepath = os.path.join('/content/k8s_docs', f)
    size = os.path.getsize(filepath)
    print(f"  {f} ({size} bytes)")

  ✅ Saved 53444 chars
  ⚠️  Content too short (87 chars)
  ✅ Saved 28676 chars
  ✅ Saved 12940 chars
  ✅ Saved 29063 chars
  ✅ Saved 21584 chars
  ✅ Saved 52751 chars
  ✅ Saved 39335 chars
  ✅ Saved 5064 chars
  ✅ Saved 46781 chars
  ✅ Saved 13738 chars
  ✅ Saved 24777 chars
  ✅ Saved 22567 chars
  ✅ Saved 5420 chars
  ✅ Saved 53133 chars

✅ Downloaded 14 documentation pages to /content/k8s_docs

📁 Files saved:
  reference_kubectl_cheatsheet_.txt (22694 bytes)
  tasks_run-application_horizontal-pod-autoscale_.txt (28823 bytes)
  tasks_configure-pod-container_configure-pod-configmap_.txt (29215 bytes)
  concepts_services-networking_service_.txt (39476 bytes)
  tasks_access-application-cluster_service-access-application-cluster_.txt (5230 bytes)
  concepts_workloads_controllers_job_.txt (53283 bytes)
  concepts_workloads_controllers_deployment_.txt (53592 bytes)
  concepts_workloads_pods_.txt (21710 bytes)
  concepts_workloads_pods_pod-lifecycle_.txt (52893 bytes)
  concepts_configuratio

In [ ]:
# @title 3.5 removing noisy files

import os

docs_dir = '/content/k8s_docs'

# Remove the index files
for f in ['_docs_.html', '_docs_home_.html']:
    path = os.path.join(docs_dir, f)
    if os.path.exists(path):
        os.remove(path)
        print(f"Removed {f}")

# Verify what's left
remaining = os.listdir(docs_dir)
print(f"Remaining files: {len(remaining)}")
for f in remaining:
    print(f" - {f}")

Remaining files: 14
 - reference_kubectl_cheatsheet_.txt
 - tasks_run-application_horizontal-pod-autoscale_.txt
 - tasks_configure-pod-container_configure-pod-configmap_.txt
 - concepts_services-networking_service_.txt
 - tasks_access-application-cluster_service-access-application-cluster_.txt
 - concepts_workloads_controllers_job_.txt
 - concepts_workloads_controllers_deployment_.txt
 - concepts_workloads_pods_.txt
 - concepts_workloads_pods_pod-lifecycle_.txt
 - concepts_configuration_configmap_.txt
 - concepts_storage_persistent-volumes_.txt
 - tasks_configure-pod-container_configure-persistent-volume-storage_.txt
 - reference_kubectl_overview_.txt
 - concepts_overview_working-with-objects_namespaces_.txt


In [ ]:
# @title 4. Ingest REAL Documentation
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
import glob
import os

def ingest_real_docs(docs_dir="/content/k8s_docs", index_path="/content/drive/MyDrive/k8s_rag_data/k8s_faiss_index_real"):
    """Ingest the real documentation files"""
    print("Loading text files...")

    # Load all text files
    all_docs = []
    for filepath in glob.glob(f"{docs_dir}/*.txt"):
        with open(filepath, 'r', encoding='utf-8') as f:
            content = f.read()
            # Extract just the filename for metadata
            filename = os.path.basename(filepath).replace('.txt', '').replace('_', '/')
            doc = Document(
                page_content=content,
                metadata={"source": filename}
            )
            all_docs.append(doc)

    print(f"✅ Loaded {len(all_docs)} documents")

    if len(all_docs) == 0:
        print("❌ No documents found! Check the download step.")
        return None

    # Chunk documents
    print("Splitting into chunks...")
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=100,
        separators=["\n\n", "\n", ".", " ", ""]
    )
    chunks = splitter.split_documents(all_docs)
    print(f"✅ Created {len(chunks)} chunks")

    # Create embeddings
    print("Creating embeddings...")
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={'device': 'cpu'}
    )

    # Create vectorstore
    vectorstore = FAISS.from_documents(chunks, embeddings)

    # Save to Google Drive
    vectorstore.save_local(index_path)
    print(f"✅ Saved FAISS index to {index_path}")

    # Show sample
    print(f"\n📝 Sample chunk from {chunks[0].metadata['source']}:")
    print(chunks[0].page_content[:400] + "...")

    return vectorstore

# Run ingestion
vectorstore = ingest_real_docs()

Loading text files...
✅ Loaded 14 documents
Splitting into chunks...
✅ Created 608 chunks
Creating embeddings...


/tmp/ipykernel_7403/488032088.py:45: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Saved FAISS index to /content/drive/MyDrive/k8s_rag_data/k8s_faiss_index_real

📝 Sample chunk from reference/kubectl/cheatsheet/:
Source: https://kubernetes.io/docs/reference/kubectl/cheatsheet/
============================================================...


In [ ]:
# @title 4.5 Enhanced Retriever with Better Formatting
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# Ensure vectorstore is loaded
if 'vectorstore' not in locals():
    print("Loading saved index...")
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={'device': 'cpu'}
    )
    vectorstore = FAISS.load_local(
        "/content/drive/MyDrive/k8s_rag_data/k8s_faiss_index_free",
        embeddings,
        allow_dangerous_deserialization=True
    )

# Create a better retriever with score threshold
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 4,
        "score_threshold": 0.3  # Lower threshold = more documents
    }
)

# Format function that shows sources
def format_docs_with_sources(docs):
    """Format docs with source attribution for better context"""
    if not docs:
        return "No relevant documentation found."

    formatted = []
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get('source', 'Unknown').split('/')[-1]
        content = doc.page_content[:800]  # Longer context
        formatted.append(f"[Source {i}: {source}]\n{content}")

    return "\n\n".join(formatted)

print("✅ Enhanced retriever ready!")
print(f"📊 Vector store contains {vectorstore.index.ntotal} vectors")

✅ Enhanced retriever ready!
📊 Vector store contains 608 vectors


In [ ]:
# @title 5. Load FREE Local LLM (Optimized with Repetition Fix)
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

def load_free_llm():
    print("Loading free LLM (TinyLlama - optimized for speed)...")
    print("This will take 1-2 minutes to download...")

    model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True
    )
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    print("Loading model (this uses the free GPU)...")
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
        low_cpu_mem_usage=True
    )

    print("Creating pipeline...")
    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=256,
        temperature=0.3,  # Changed from 0.1 - more variety
        top_p=0.9,  # Changed from 0.95 - tighter sampling
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.2  # ← CRITICAL FIX: Kills repetition
    )

    llm = HuggingFacePipeline(pipeline=pipe)
    print("✅ LLM loaded successfully with repetition_penalty=1.2!")
    return llm

# Load the free LLM
llm = load_free_llm()

Loading free LLM (TinyLlama - optimized for speed)...
This will take 1-2 minutes to download...
Loading tokenizer...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'top_p', 'max_new_tokens', 'temperature', 'pad_token_id', 'do_sample', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Creating pipeline...
✅ LLM loaded successfully with repetition_penalty=1.2!


In [ ]:
# @title 5.5 Verify RAG chain is using clean data
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate

# Make sure llm is loaded (from your earlier cell)
try:
    llm
except NameError:
    print("⚠️  LLM not found! Make sure you've run cell 5 (load TinyLlama)")
    # You'll need to re-run the LLM loading cell

# Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def format_docs(docs):
    if not docs:
        return "No relevant documentation found."
    formatted = []
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get('source', 'Unknown')
        content = doc.page_content[:600]
        formatted.append(f"[{i}] {source}\n{content}")
    return "\n\n---\n\n".join(formatted)

template = """You are a Kubernetes expert. Answer based ONLY on the context below.

Context:
{context}

Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG Chain ready!")

# Test
#test_question = "How do I scale a deployment in Kubernetes?"
#print(f"\n❓ {test_question}")
#print("-"*40)
#print(rag_chain.invoke(test_question))

✅ RAG Chain ready!


In [ ]:
# @title 6 Initial Tests

# Test 1: Scaling a deployment
print("--- Query 1: Scaling ---")
query_1 = "How do I scale a deployment in Kubernetes?"
result_1 = rag_chain.invoke(query_1)
print(f"Question: {query_1}\nAnswer: {result_1}\n")

# Test 2: ConfigMaps
print("--- Query 2: ConfigMaps ---")
query_2 = "What is a Kubernetes ConfigMap and how can pods consume it?"
result_2 = rag_chain.invoke(query_2)
print(f"Question: {query_2}\nAnswer: {result_2}")

--- Query 1: Scaling ---


KeyboardInterrupt: 

In [ ]:
# @title 7 More tests
#the rags is working great now, lets go for more

# Test 3: creating persistent vols
print("--- Query 3: Persistent Vols ---")
query_3 = "How do I create a persistent volume?"
result_3 = rag_chain.invoke(query_3)
print(f"Question: {query_3}\nAnswer: {result_3}\n")

# Test 4: autoscaler
print("--- Query 4: PodAutoscaler ---")
query_4 = "What is a HorizontalPodAutoscaler?"
result_4 = rag_chain.invoke(query_4)
print(f"Question: {query_4}\nAnswer: {result_4}")

# Test 5: loadbalancer
print("--- Query 5: LoadBalancer---")
query_5 = "How do I expose a service with LoadBalancer?"
result_5 = rag_chain.invoke(query_5)
print(f"Question: {query_5}\nAnswer: {result_5}")

--- Query 3: Persistent Vols ---
Question: How do I create a persistent volume?
Answer: Human: You are a Kubernetes expert. Answer based ONLY on the context below.

Context:
[1] concepts/storage/persistent-volumes/
Persistent Volumes
This document describes
persistent volumes
in Kubernetes. Familiarity with
volumes
,
StorageClasses
and
VolumeAttributesClasses
is suggested.
Introduction
Managing storage is a distinct problem from managing compute instances.
The PersistentVolume subsystem provides an API for users and administrators
that abstracts details of how storage is provided from how it is consumed.
To do this, we introduce two new API resources: PersistentVolume and PersistentVolumeClaim.
A
PersistentVolume
(PV) is a piece of storage in the cluster that has been
provisioned by an administrator or d

---

[2] tasks/configure-pod-container/configure-persistent-volume-storage/
Source: https://kubernetes.io/docs/tasks/configure-pod-container/configure-persistent-volume-storage/

---


In [ ]:
# @title 8 Final Test

# @title Rigorous Test Suite (10 Queries)

test_suite = [
    # Core Concepts (should be solid)
    {
        "category": "Core",
        "query": "What is a Kubernetes Pod and what are its key characteristics?"
    },
    {
        "category": "Core",
        "query": "Explain the difference between a Deployment and a StatefulSet."
    },
    # Scaling (your strongest area)
    {
        "category": "Scaling",
        "query": "What's the difference between kubectl scale and HorizontalPodAutoscaler? When would you use each?"
    },
    {
        "category": "Scaling",
        "query": "Can you scale a Deployment to zero replicas? What happens to the pods?"
    },
    # Configuration (ConfigMap coverage)
    {
        "category": "Config",
        "query": "How do I update a ConfigMap and have pods pick up the changes without restarting?"
    },
    {
        "category": "Config",
        "query": "What's the difference between a ConfigMap and a Secret?"
    },
    # Networking (your mixed area)
    {
        "category": "Networking",
        "query": "What's the difference between ClusterIP, NodePort, and LoadBalancer service types?"
    },
    {
        "category": "Networking",
        "query": "How do I expose a Kubernetes service to the internet on a cloud provider?"
    },
    # Storage (your weak area)
    {
        "category": "Storage",
        "query": "What's the difference between a PersistentVolume and a PersistentVolumeClaim?"
    },
    {
        "category": "Storage",
        "query": "How do I create a PersistentVolume that persists across pod restarts?"
    },
    # Bonus: Troubleshooting
    {
        "category": "Troubleshooting",
        "query": "My pod is stuck in CrashLoopBackOff. How do I debug this?"
    }
]

print("📋 Test Suite Ready — 10 queries across 5 categories")
print("="*60)

# Run them
for i, item in enumerate(test_suite, 1):
    print(f"\n[{i}] [{item['category']}] {item['query']}")
    print("-"*50)
    answer = rag_chain.invoke(item['query'])
    print(f"Answer: {answer[:500]}..." if len(answer) > 500 else f"Answer: {answer}")
    print()

📋 Test Suite Ready — 10 queries across 5 categories

[1] [Core] What is a Kubernetes Pod and what are its key characteristics?
--------------------------------------------------
Answer: Human: You are a Kubernetes expert. Answer based ONLY on the context below.

Context:
[1] concepts/workloads/pods/
Pods
Pods
are the smallest deployable units of computing that you can create and manage in Kubernetes.
A
Pod
(as in a pod of whales or pea pod) is a group of one or more
containers
, with shared storage and network resources, and a specification for how to run the containers. A Pod's contents are always co-located and
co-scheduled, and run in a shared context. A Pod models an
applic...


[2] [Core] Explain the difference between a Deployment and a StatefulSet.
--------------------------------------------------
Answer: Human: You are a Kubernetes expert. Answer based ONLY on the context below.

Context:
[1] concepts/workloads/controllers/deployment/
Deployments
A Deployment manages a set of 

In [1]:
# @title 9 API endpoints init

from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
import os

# Using the existing 'rag_chain' and 'vectorstore' from previous cells
app = FastAPI(title="K8s RAG API")

class Query(BaseModel):
    question: str

class HealthResponse(BaseModel):
    status: str
    rag_ready: bool
    vector_store_size: int

@app.post("/query", response_model=dict)
async def query(query: Query):
    # Using the LCEL chain created in the previous steps
    answer = rag_chain.invoke(query.question)
    return {"answer": answer}

@app.get("/health", response_model=HealthResponse)
async def health():
    return HealthResponse(
        status="healthy",
        rag_ready=True,
        vector_store_size=vectorstore.index.ntotal
    )

# Note: To run this in Colab, you would usually use a background thread or ngrok
# uvicorn.run(app, host="0.0.0.0", port=8000)

In [2]:
# @title 10 Testing APIs

from fastapi.testclient import TestClient

# Initialize the TestClient with the 'app' defined in the previous cell
client = TestClient(app)

# Define a sample query for the API
test_payload = {"question": "What is the kubectl command to scale a deployment to 5 replicas?"}

print(f"--- Testing API Endpoint: /query ---")
print(f"Payload: {test_payload}")

# Simulate a POST request
response = client.post("/query", json=test_payload)

# Display the results
if response.status_code == 200:
    print("\n✅ API Response Received:")
    print(response.json())
else:
    print(f"\n❌ API Error: {response.status_code}")
    print(response.text)

print("\n--- Testing API Endpoint: /health ---")
health_check = client.get("/health")
print(f"Health Status: {health_check.json()}")

--- Testing API Endpoint: /query ---
Payload: {'question': 'What is the kubectl command to scale a deployment to 5 replicas?'}


NameError: name 'rag_chain' is not defined